# 04 -- User Profiles

## What We'll Cover
1. Static vs Dynamic profiles
2. How profiles auto-build from ingestion
3. Using profiles for context injection
4. Profile + Search combined (the recommended pattern)
5. Profile-driven use cases (chatbots, support, education, dev tools)


## 4.1 Why Profiles?

Traditional memory systems rely entirely on search. But search is **too narrow** -- when you search for "project updates", you miss that the user prefers bullet points, works in PST, and uses specific terminology.

Profiles provide **the foundation**: a complete picture of who the user is -- always available, no search needed.

**Performance:**
- Profile call: ~50ms
- Search call: 200-500ms
- Context retrieval: 1 call instead of 3-5


## 4.2 Static vs Dynamic Profiles

### Static Profile (Long-term, Stable Facts)
- "Sarah is a senior software engineer at TechCorp"
- "Sarah specializes in distributed systems"
- "Sarah prefers technical docs over video tutorials"

### Dynamic Profile (Recent Context, Temporary States)
- "Sarah is migrating the payment service to microservices"
- "Sarah is preparing for a conference talk next month"
- "Sarah is debugging a memory leak in auth service"

Profiles separate these automatically. Static facts persist until contradicted. Dynamic facts evolve as the user's context changes.


In [ ]:
# 4.3 Basic profile retrieval
from supermemory import Supermemory
client = Supermemory()

# Get profile without a search query
profile = client.profile(container_tag="user_sarah")

print("=== STATIC PROFILE ===")
for fact in profile.profile.static:
    print(f"  - {fact}")

print(f"\n=== DYNAMIC PROFILE ===")
for fact in profile.profile.dynamic:
    print(f"  - {fact}")


In [ ]:
# 4.4 Profile with search -- the recommended pattern
# profile() + query = context + search in ONE call (~50ms)
profile = client.profile(
    container_tag="user_sarah",
    q="What design tools does Sarah use?",  # search query
    threshold=0.5  # relevance threshold
)

print("=== Static Facts ===")
for fact in profile.profile.static:
    print(f"  - {fact}")

print(f"\n=== Dynamic Facts ===")
for fact in profile.profile.dynamic:
    print(f"  - {fact}")

print(f"\n=== Relevant Search Memories ===")
for r in profile.search_results.results[:5]:
    score = r.get("score", "N/A")
    memory = r.get("memory", str(r))
    print(f"  [{score}] {memory[:100]}...")


In [ ]:
# 4.5 Building context for an LLM from a profile
# This is the core chatbot/agent pattern
def build_llm_context(profile, user_name="User"):
    """Build a system prompt from a Supermemory profile."""
    static = "\n".join(f"- {f}" for f in profile.profile.static)
    dynamic = "\n".join(f"- {f}" for f in profile.profile.dynamic)
    memories = "\n".join(
        f"- {r.get('memory', str(r))}"
        for r in profile.search_results.results[:5]
    )

    system_prompt = f"""You are assisting {user_name}.

Background (long-term):
{static}

Current focus (recent):
{dynamic}

Relevant past context:
{memories}

Adjust your responses to their expertise, preferences, and current work."""
    return system_prompt

# Example usage
prompt = build_llm_context(profile, user_name="Sarah")
print(prompt)
print("\n--- Now feed this system prompt to your LLM! ---")


In [ ]:
# 4.6 Profiles self-build -- just keep adding content
# Add more conversations and the profile updates automatically
more_conversations = [
    "user: I finished the dashboard design! It uses a 12-column grid.\nassistant: Great work!",
    "user: I prefer dark mode for all my design tools.\nassistant: Noted.",
    "user: The Figma component library needs an update for accessibility.\nassistant: I can help.",
]

for conv in more_conversations:
    client.add(content=conv, container_tag="user_sarah")
    print(f"Added: {conv[:60]}...")

# Check updated profile
updated = client.profile(container_tag="user_sarah")
print(f"\nStatic facts: {len(updated.profile.static)}")
print(f"Dynamic facts: {len(updated.profile.dynamic)}")
print("New facts have been extracted and merged into the profile!")


## 4.7 Use Cases

### Personalized AI Assistants
Profiles provide: expertise level, communication preferences, tools used, current projects.

### Customer Support
Profiles provide: product usage, previous issues, tech proficiency. No more "let me look up your account."

### Educational Platforms
Profiles provide: learning style, completed courses, strengths/weaknesses.

### Development Tools
Profiles provide: preferred languages, coding style, current project context.

## 4.8 Key Takeaways

- Profiles auto-build from ingestion -- no manual management needed
- Static = long-term stable facts, Dynamic = recent context
- `profile()` + query = context + search in one ~50ms call
- Use profiles to build rich system prompts for LLMs
- The Profile -> Chat -> Store pattern is the foundation of every integration

**Next:** Notebook 05 -- Graph Memory & Relationships
